# Canadian Housing Market Dashboard — Data Preparation

This notebook takes the original CREA MLS® HPI Excel file (60 separate sheets) and flattens it into a single clean CSV file ready for Tableau Public.

**Before running:**
- Make sure your folder structure looks like this:
```
Canadian Housing Market Dashboard/
├── prepare_data.ipynb        ← this notebook
└── Dataset/
    └── Not Seasonally Adjusted (M).xlsx
```
- Place this notebook inside the `Canadian Housing Market Dashboard` folder
- Then run all cells top to bottom

**Output:** `Dataset/Canadian_HPI_Flat.csv` — connect this file directly in Tableau Public

In [1]:
import pandas as pd
import os

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# ── Path setup ────────────────────────────────────────────────────────────────
# os.getcwd() picks up the folder where this notebook is saved
# No need to change anything — just make sure the notebook is in the right folder

base_dir    = os.getcwd()
input_file  = os.path.join(base_dir, 'Dataset', 'Not Seasonally Adjusted (M).xlsx')
output_file = os.path.join(base_dir, 'Dataset', 'Canadian_HPI_Flat.csv')

print(f'Looking for Excel file at:')
print(f'  {input_file}')
print()

if os.path.exists(input_file):
    print('✅ File found! Ready to proceed.')
else:
    print('❌ File not found. Check that:')
    print('   1. This notebook is inside the Canadian Housing Market Dashboard folder')
    print('   2. The Excel file is inside a subfolder called Dataset')

Looking for Excel file at:
  C:\Users\hedye\Desktop\Drive D\Courses I took\GitHub\data-analytics-portfolio\Tableau\Canadian Housing Market Dashboard\Dataset\Not Seasonally Adjusted (M).xlsx

✅ File found! Ready to proceed.


In [3]:
# ── Province lookup ───────────────────────────────────────────────────────────
# Maps each sheet name to its province abbreviation

province_map = {
    'BRITISH_COLUMBIA': 'BC', 'VANCOUVER_ISLAND': 'BC', 'VICTORIA': 'BC',
    'LOWER_MAINLAND': 'BC', 'GREATER_VANCOUVER': 'BC', 'FRASER_VALLEY': 'BC',
    'CHILLIWACK_AND_DISTRICT': 'BC', 'INTERIOR_BC': 'BC',
    'ALBERTA': 'AB', 'CALGARY': 'AB', 'EDMONTON': 'AB',
    'SASKATCHEWAN': 'SK', 'REGINA': 'SK', 'SASKATOON': 'SK',
    'WINNIPEG': 'MB',
    'ONTARIO': 'ON', 'BANCROFT_AND_AREA': 'ON', 'BARRIE_AND_DISTRICT': 'ON',
    'BRANTFORD_REGION': 'ON', 'CAMBRIDGE': 'ON', 'GREY_BRUCE_OWEN_SOUND': 'ON',
    'GUELPH_AND_DISTRICT': 'ON', 'HAMILTON_BURLINGTON': 'ON', 'HURON_PERTH': 'ON',
    'KAWARTHA_LAKES': 'ON', 'KINGSTON_AND_AREA': 'ON', 'KITCHENER_WATERLOO': 'ON',
    'LAKELANDS': 'ON', 'LONDON_ST_THOMAS': 'ON', 'MISSISSAUGA': 'ON',
    'NIAGARA_REGION': 'ON', 'NORTH_BAY': 'ON', 'NORTHUMBERLAND_HILLS': 'ON',
    'OAKVILLE_MILTON': 'ON', 'OTTAWA': 'ON', 'PETERBOROUGH_AND_KAWARTHAS': 'ON',
    'QUINTE_AND_DISTRICT': 'ON', 'RIDEAU_ST_LAWRENCE': 'ON', 'SAULT_STE_MARIE': 'ON',
    'SIMCOE_AND_DISTRICT': 'ON', 'SUDBURY': 'ON', 'GREATER_TORONTO': 'ON',
    'WINDSOR_ESSEX': 'ON', 'WOODSTOCK_INGERSOLL_TILLSONBURG': 'ON',
    'QUEBEC': 'QC', 'CENTRE_DU_QUEBEC': 'QC', 'ESTRIE': 'QC', 'MAURICIE': 'QC',
    'MONTREAL_CMA': 'QC', 'QUEBEC_CMA': 'QC',
    'NEW_BRUNSWICK': 'NB', 'FREDERICTON': 'NB', 'GREATER_MONCTON': 'NB', 'SAINT_JOHN_NB': 'NB',
    'NOVA_SCOTIA': 'NS', 'HALIFAX_DARTMOUTH': 'NS',
    'PRINCE_EDWARD_ISLAND': 'PE',
    'NEWFOUNDLAND_AND_LABRADOR': 'NL', 'ST_JOHNS_NL': 'NL',
}

# ── Clean region display names ────────────────────────────────────────────────
region_name_map = {k: k.replace('_', ' ').title() for k in province_map}
region_name_map.update({
    'GREATER_VANCOUVER': 'Greater Vancouver',
    'GREATER_TORONTO': 'Greater Toronto',
    'GREATER_MONCTON': 'Greater Moncton',
    'HALIFAX_DARTMOUTH': 'Halifax-Dartmouth',
    'MONTREAL_CMA': 'Montreal CMA',
    'QUEBEC_CMA': 'Quebec City CMA',
    'ST_JOHNS_NL': 'St. Johns NL',
    'SAINT_JOHN_NB': 'Saint John NB',
    'WOODSTOCK_INGERSOLL_TILLSONBURG': 'Woodstock-Ingersoll-Tillsonburg',
    'LONDON_ST_THOMAS': 'London-St. Thomas',
    'GREY_BRUCE_OWEN_SOUND': 'Grey Bruce Owen Sound',
    'CHILLIWACK_AND_DISTRICT': 'Chilliwack And District',
})

print(f'Province and region mappings ready for {len(province_map)} regions.')

Province and region mappings ready for 59 regions.


In [4]:
# ── Read and combine all sheets ───────────────────────────────────────────────
# Note: Some sheets are missing Townhouse or Apartment columns
# We fill those with null so all regions have the same structure

all_cols = [
    'Date', 'Composite_HPI', 'Single_Family_HPI', 'One_Storey_HPI', 'Two_Storey_HPI',
    'Townhouse_HPI', 'Apartment_HPI', 'Composite_Benchmark', 'Single_Family_Benchmark',
    'One_Storey_Benchmark', 'Two_Storey_Benchmark', 'Townhouse_Benchmark', 'Apartment_Benchmark',
]

print('Reading Excel file — this may take 10-20 seconds...')
xl = pd.read_excel(input_file, sheet_name=None)
print(f'Found {len(xl)} sheets.')

frames = []
for sheet, df in xl.items():
    if sheet == 'AGGREGATE':
        continue   # AGGREGATE is the national summary — kept as a separate file
    df = df.copy()
    for col in all_cols:
        if col not in df.columns:
            df[col] = None   # fill missing columns with null
    df = df[all_cols]
    df['Region']     = region_name_map.get(sheet, sheet.replace('_', ' ').title())
    df['Province']   = province_map.get(sheet, 'Unknown')
    df['Region_Raw'] = sheet
    frames.append(df)

print(f'Processed {len(frames)} regional sheets.')

Reading Excel file — this may take 10-20 seconds...
Found 60 sheets.
Processed 59 regional sheets.


In [5]:
# ── Final assembly and save ───────────────────────────────────────────────────

result = pd.concat(frames, ignore_index=True)
result['Date'] = pd.to_datetime(result['Date'])
result = result.sort_values(['Region', 'Date']).reset_index(drop=True)

col_order = ['Date', 'Region', 'Province', 'Region_Raw'] + all_cols[1:]
result = result[col_order]

result.to_csv(output_file, index=False)

print('✅ Done!')
print(f'   Rows     : {len(result):,}')
print(f'   Regions  : {result["Region"].nunique()}')
print(f'   Provinces: {result["Province"].nunique()}')
print(f'   Date range: {result["Date"].min().date()} → {result["Date"].max().date()}')
print(f'\n   Output saved to:')
print(f'   {output_file}')
print(f'\n   → Open Canadian_HPI_Flat.csv in Tableau Public to start building!')

✅ Done!
   Rows     : 14,986
   Regions  : 59
   Provinces: 10
   Date range: 2005-01-01 → 2026-02-01

   Output saved to:
   C:\Users\hedye\Desktop\Drive D\Courses I took\GitHub\data-analytics-portfolio\Tableau\Canadian Housing Market Dashboard\Dataset\Canadian_HPI_Flat.csv

   → Open Canadian_HPI_Flat.csv in Tableau Public to start building!


C:\Users\hedye\AppData\Local\Temp\ipykernel_23432\1593557414.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat(frames, ignore_index=True)
